<a href="https://colab.research.google.com/github/LleilaA13/FDS25-26/blob/main/classnotes/grid_search_KfoldCV" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Grid search

This notebook was originally inspired by [here](https://colab.research.google.com/github/goodboychan/chans_jupyter/blob/main/_notebooks/2020-08-05-02-Grid-search.ipynb).


In [ ]:
import pandas as pd
import numpy as np
from pprint import pprint

## Introducing Grid Search

### Build Grid Search functions

In data science, it is extremely valuable to rebuild standard workflows *from scratch* to gain a deeper understanding of what happens behind the scenes.

Even though libraries like `scikit-learn` provide ready-to-use tools such as `GridSearchCV`, manually implementing grid search helps clarify how:

1. Hyperparameter combinations are generated  
2. Models are trained and evaluated  
3. Performance metrics are compared  

In this example, we will build a simple grid search function to explore two hyperparameters of a **Logistic Regression** model:
- `C` (inverse of regularization strength)  
- `penalty` (type of regularization applied)

In [ ]:
from sklearn.datasets import load_breast_cancer
import pandas as pd

# Load dataset from sklearn
data = load_breast_cancer()

# Convert to pandas DataFrame for easier manipulation
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

# Visualize the first 5 rows of the dataset
X.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


In [ ]:
from sklearn.model_selection import train_test_split

# Split into training and test sets (70% / 30%)
# random_state=42 ensures reproducibility
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, shuffle=True, random_state=42
)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Function to train and evaluate a Logistic Regression model for given hyperparameters
def logreg_grid_search(C_value, penalty_type):
    """
    Trains and evaluates a Logistic Regression model using the specified hyperparameters.

    Parameters
    ----------
    C_value : float
        Inverse of regularization strength.
        Smaller values imply stronger regularization (simpler model).
        Larger values imply weaker regularization (risk of overfitting).

    penalty_type : str
        Type of regularization to apply:
        - 'l1' → Lasso (sparse coefficients)
        - 'l2' → Ridge (smaller but non-zero coefficients)

    Returns
    -------
    list
        [C_value, penalty_type, accuracy]
        Contains the hyperparameter values tested and the resulting accuracy on the test set.
    """
    # Note: C is the *inverse* of regularization strength
    # Low values --> strong regularization
    # High values --> risk of overfitting
    model = LogisticRegression(C=C_value, penalty=penalty_type, solver="liblinear")

    # Train and predict
    predictions = model.fit(X_train, y_train).predict(X_test)

    # Return tested hyperparameters and accuracy
    return [C_value, penalty_type, accuracy_score(y_test, predictions)]



### Iteratively tune multiple hyperparameters

Now that we have defined the `logreg_grid_search` function,  
we can loop over different values of the hyperparameters to test multiple combinations.  

In this case, we will explore:
- different values of `C` (regularization strength)
- different types of `penalty` (`l1` and `l2`)

In [ ]:
from pprint import pprint

# Create the lists of hyperparameter values to test
results_list = []
C_values = [0.01, 0.1, 1, 10]
penalty_types = ["l1", "l2"]

# Test all combinations
for C_value in C_values:
    for penalty_type in penalty_types:
        results_list.append(logreg_grid_search(C_value, penalty_type))

# Print the results
pprint(results_list)

[[0.01, 'l1', 0.9532163742690059],
 [0.01, 'l2', 0.9649122807017544],
 [0.1, 'l1', 0.9590643274853801],
 [0.1, 'l2', 0.9766081871345029],
 [1, 'l1', 0.9649122807017544],
 [1, 'l2', 0.9649122807017544],
 [10, 'l1', 0.9707602339181286],
 [10, 'l2', 0.9649122807017544]]


### Extending the grid search function

We can now extend our grid search function to include a **third hyperparameter**.  
For Logistic Regression, besides `C` and `penalty`, we can also tune the **solver** (the optimization algorithm used).  

Different solvers support different penalties:
- `liblinear` → supports both `l1` and `l2`
- `lbfgs` → supports only `l2`

In [ ]:
def logreg_grid_search_extended(C_value, penalty_type, solver_type):
    # Create the model using the three hyperparameters
    model = LogisticRegression(C=C_value, penalty=penalty_type, solver=solver_type, max_iter=1000)

    # Train and predict
    predictions = model.fit(X_train, y_train).predict(X_test)

    # Return hyperparameters and score
    return [C_value, penalty_type, solver_type, accuracy_score(y_test, predictions)]

In [ ]:
# Define the hyperparameter grids
results_list = []
C_values = [0.01, 0.1, 1, 10]
penalty_types = ["l1", "l2"]
solver_types = ["liblinear", "lbfgs"]

# Iterate over all combinations
for C_value in C_values:
    for penalty_type in penalty_types:
        for solver_type in solver_types:
            try:
                results_list.append(logreg_grid_search_extended(C_value, penalty_type, solver_type))
            except ValueError:
                # Skip invalid combinations (e.g., l1 with lbfgs)
                continue

# Print all results
pprint(results_list)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _c

[[0.01, 'l1', 'liblinear', 0.9532163742690059],
 [0.01, 'l2', 'liblinear', 0.9649122807017544],
 [0.01, 'l2', 'lbfgs', 0.9649122807017544],
 [0.1, 'l1', 'liblinear', 0.9590643274853801],
 [0.1, 'l2', 'liblinear', 0.9766081871345029],
 [0.1, 'l2', 'lbfgs', 0.9649122807017544],
 [1, 'l1', 'liblinear', 0.9649122807017544],
 [1, 'l2', 'liblinear', 0.9649122807017544],
 [1, 'l2', 'lbfgs', 0.9707602339181286],
 [10, 'l1', 'liblinear', 0.9707602339181286],
 [10, 'l2', 'liblinear', 0.9649122807017544],
 [10, 'l2', 'lbfgs', 0.9649122807017544]]


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


## Grid Search with Scikit-Learn

A grid search systematically explores combinations of hyperparameters to find the configuration that maximizes a chosen performance metric.

### Steps in a Grid Search
1. Choose an algorithm (estimator) whose hyperparameters you want to tune.  
2. Define which hyperparameters to include in the search.  
3. Specify a range of possible values for each hyperparameter.  
4. Choose a cross-validation scheme to estimate model performance.  
5. Select a scoring metric to evaluate each combination.  
6. Optionally, use parallel processing and store additional information such as training scores.

### GridSearchCV with Scikit-Learn

The `GridSearchCV` module in `scikit-learn` automates all these steps efficiently.

In this example, we will:
- Use a **Logistic Regression** model.  
- Perform **5-fold cross-validation**.  
- Tune the hyperparameters:
  - `C` → [0.01, 0.1, 1, 10]  
  - `penalty` → ['l1', 'l2']  
  - `solver` → ['liblinear', 'lbfgs']  
- Use **ROC AUC** as the scoring metric.  
- Run the search in **parallel with 4 cores**.  
- **Refit** the best model automatically and return training scores.

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import LogisticRegression

# Create a Logistic Regression model (base estimator)
logreg = LogisticRegression(max_iter=1000)

# Define the parameter grid to search
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l1', 'l2'],
    'solver': ['liblinear', 'lbfgs']
}

# Create the GridSearchCV object
grid_logreg = GridSearchCV(
    estimator=logreg,
    param_grid=param_grid,
    scoring='roc_auc',
    n_jobs=4,        # use 4 cores in parallel
    cv=5,            # 5-fold cross-validation, more on this later
    refit=True,      # retrain the best model on the full training set
    return_train_score=True
)

print(grid_logreg)

GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000), n_jobs=4,
             param_grid={'C': [0.01, 0.1, 1, 10], 'penalty': ['l1', 'l2'],
                         'solver': ['liblinear', 'lbfgs']},
             return_train_score=True, scoring='roc_auc')


## Understanding a grid search output


In [ ]:
# Fit the GridSearchCV object on the training data
grid_logreg.fit(X_train, y_train)

# Read the cv_results_ property into a DataFrame
cv_results_df = pd.DataFrame(grid_logreg.cv_results_)

# Display the full results table
print(cv_results_df)

# Extract and print only the column with the tested hyperparameter combinations
column = cv_results_df.loc[:, ["params"]]
print(column)

# Extract and print the row corresponding to the best mean test score (rank = 1)
best_row = cv_results_df[cv_results_df["rank_test_score"] == 1]
print(best_row)

    mean_fit_time  std_fit_time  mean_score_time  std_score_time  param_C  \
0        0.059141      0.023191         0.021496        0.003622     0.01   
1        0.006607      0.002061         0.000000        0.000000     0.01   
2        0.026415      0.005103         0.020270        0.005696     0.01   
3        0.441315      0.062541         0.020797        0.006820     0.01   
4        0.448163      0.104952         0.017048        0.005611     0.10   
5        0.007383      0.003028         0.000000        0.000000     0.10   
6        0.027335      0.010475         0.017836        0.004831     0.10   
7        1.034695      0.176653         0.025065        0.005287     0.10   
8        1.212259      0.261947         0.008080        0.002421     1.00   
9        0.003618      0.000846         0.000000        0.000000     1.00   
10       0.020992      0.004623         0.010779        0.001883     1.00   
11       0.577820      0.090345         0.014057        0.005590     1.00   

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py:528: FitFailedWarning: 
20 fits failed out of a total of 80.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
20 fits failed with the following error:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "/usr/local/lib/python3.12/dist-packages/sklearn/base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py", line 1193, in fit
    solver = _check_solver

### 🧩 Interpreting the Grid Search Results

From the DataFrame above, each row represents one combination of hyperparameters tested.

Key observations:

- **`param_C`, `param_penalty`, `param_solver`** show the specific hyperparameter values used for each run.  
- **`mean_test_score`** is the average ROC AUC across the 5 cross-validation folds — this is the main metric to compare models.  
- **`rank_test_score`** orders the models by performance (1 = best).  
- **`mean_train_score`** and **`std_train_score`** indicate how well the model fits the training data and the variability across folds.  
- **`mean_fit_time`** and **`mean_score_time`** report the average computation times.

---

### 🏆 Best Model

The **best performing model** has:
- `C = 10`
- `penalty = 'l1'`
- `solver = 'liblinear'`
- `mean_test_score ≈ 0.9898` (highest ROC AUC)

This means:
- A **weaker regularization** (`C=10`) allowed slightly better performance.
- The **L1 penalty** encouraged sparsity (some coefficients set to zero), which helped generalization.
- `liblinear` was compatible with L1 and converged efficiently.

---

### 💡 Conclusion

Grid search systematically tested all valid parameter combinations.  
By inspecting `cv_results_`, we can directly identify:
- The **best configuration** (`rank_test_score == 1`)  
- The **generalization quality** (high `mean_test_score`, stable `std_test_score`)  
- The **training vs testing gap**, useful to detect overfitting.


### Analyzing the best results
At the end of the day, we primarily care about the best performing 'square' in a grid search. Luckily Scikit Learn's `gridSearchCV` objects have a number of parameters that provide key information on just the best square (or row in `cv_results_`).

Three properties you will explore are:

- `best_score_` – The score (here ROC_AUC) from the best-performing square.
- `best_index_` – The index of the row in `cv_results_` containing information on the best-performing square.
- `best_params_` – A dictionary of the parameters that gave the best score, for example 'max_depth': 10

It's not easy to debug this output, right? Fortunately, we can do something like this:

In [ ]:
# Print the best ROC_AUC score found during grid search
best_score = grid_logreg.best_score_
print("Best ROC_AUC score:", best_score)

# Create a DataFrame from cv_results_ and extract the best-performing row
cv_results_df = pd.DataFrame(grid_logreg.cv_results_)
best_row = cv_results_df.loc[[grid_logreg.best_index_]]
print("\nBest row in cv_results_:")
print(best_row)

# Extract the best hyperparameter combination
best_params = grid_logreg.best_params_
print("\nBest hyperparameters:")
print(best_params)


Best ROC_AUC score: 0.9898231292517007

Best row in cv_results_:
    mean_fit_time  std_fit_time  mean_score_time  std_score_time  param_C  \
12       1.020559      0.362423         0.010399        0.002224     10.0   

   param_penalty param_solver  \
12            l1    liblinear   

                                               params  split0_test_score  \
12  {'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}           0.998667   

    split1_test_score  ...  mean_test_score  std_test_score  rank_test_score  \
12           0.961333  ...         0.989823        0.014816                1   

    split0_train_score  split1_train_score  split2_train_score  \
12            0.995313             0.99924            0.994637   

    split3_train_score  split4_train_score  mean_train_score  std_train_score  
12            0.995812            0.996597           0.99632         0.001594  

[1 rows x 23 columns]

Best hyperparameters:
{'C': 10, 'penalty': 'l1', 'solver': 'liblinear'}


### Using the Best Results

After running a grid search, the optimal model is stored in the `.best_estimator_` property.  
We can use this model directly to make predictions on the test set and compute evaluation metrics such as the confusion matrix and ROC-AUC score.

Using `predict_proba` instead of `predict` allows us to obtain probabilities, which are essential for computing the ROC-AUC metric.

In [ ]:
from sklearn.metrics import confusion_matrix, roc_auc_score

# Check what type of object the best_estimator_ property returns
print(type(grid_logreg.best_estimator_))

# Use the best estimator to make predictions on the test set
predictions = grid_logreg.best_estimator_.predict(X_test)

# Display a few predicted class labels (0 = benign, 1 = malignant)
print("Predicted labels:", predictions[:10])

# Create and display the confusion matrix
print("Confusion Matrix:\n", confusion_matrix(y_test, predictions))

# Compute and display the ROC-AUC score using predicted probabilities
predictions_proba = grid_logreg.best_estimator_.predict_proba(X_test)[:, 1]
print("ROC-AUC Score:", roc_auc_score(y_test, predictions_proba))

<class 'sklearn.linear_model._logistic.LogisticRegression'>
Predicted labels: [1 0 0 1 1 0 0 0 1 1]
Confusion Matrix:
 [[ 61   2]
 [  3 105]]
ROC-AUC Score: 0.996031746031746


The `.best_estimator_` property is a really powerful property to understand for streamlining your machine learning model building process. You now can run a grid search and seamlessly use the best model from that search to make predictions.

---

# PART 2 - Understanding K-Fold Cross-Validation


In supervised machine learning, we need to evaluate how well a model generalizes to unseen data.  
A single train/test split can be misleading if the data are not uniformly distributed.

**K-Fold Cross-Validation** provides a more reliable estimate of model performance by repeatedly splitting the data.

---

### How K-Fold Cross-Validation Works
1. The dataset is divided into **K equal folds** (subsets).
2. The model is trained on **K−1 folds** and tested on the **remaining one**.
3. This process repeats **K times**, each time using a different fold as the test set.
4. The final score is the **average** of the K test scores.

This ensures that **every observation is used for both training and testing** exactly once, providing a robust estimate of model performance.


In [ ]:
# Import core libraries
import numpy as np
import pandas as pd
from sklearn.datasets import load_wine
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import KFold, cross_val_score

In [ ]:
# Load the Wine dataset (multi-class classification)
data = load_wine()

# Convert to DataFrame for readability
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="class")

# Display first few rows
X.head()

,alcohol,malic_acid,ash,alcalinity_of_ash,magnesium,total_phenols,flavanoids,nonflavanoid_phenols,proanthocyanins,color_intensity,hue,od280/od315_of_diluted_wines,proline
0,14.23,1.71,2.43,15.6,127.0,2.80,3.06,0.28,2.29,5.64,1.04,3.92,1065.0
1,13.20,1.78,2.14,11.2,100.0,2.65,2.76,0.26,1.28,4.38,1.05,3.40,1050.0
2,13.16,2.36,2.67,18.6,101.0,2.80,3.24,0.30,2.81,5.68,1.03,3.17,1185.0
3,14.37,1.95,2.50,16.8,113.0,3.85,3.49,0.24,2.18,7.80,0.86,3.45,1480.0
4,13.24,2.59,2.87,21.0,118.0,2.80,2.69,0.39,1.82,4.32,1.04,2.93,735.0


In [ ]:
# Define K-Fold cross-validator
# n_splits = number of folds
# shuffle=True to randomize data before splitting
# random_state ensures reproducibility
kf = KFold(n_splits=5, shuffle=True, random_state=42)

# Define the model
logreg = LogisticRegression(max_iter=1000)

In [ ]:
from sklearn.metrics import accuracy_score, make_scorer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

# Logistic Regression often benefits from feature scaling.
# We'll use a pipeline: first standardize the data, then apply logistic regression.
pipeline = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))

# Perform 5-fold cross-validation using accuracy as the metric
scores = cross_val_score(pipeline, X, y, cv=kf, scoring='accuracy')

# Display individual fold scores and summary statistics
print("Accuracy per fold:", np.round(scores, 4))
print("Mean accuracy:", np.round(scores.mean(), 4))
print("Standard deviation:", np.round(scores.std(), 4))

Accuracy per fold: [1.     0.9722 1.     0.9714 1.    ]
Mean accuracy: 0.9887
Standard deviation: 0.0138


### 🧩 Interpreting the Cross-Validation Results

Each value in the array represents the **accuracy** obtained on one of the five validation folds.

- **Accuracy per fold:** [1.0000, 0.9722, 1.0000, 0.9714, 1.0000]  
  The model performs very consistently across all folds.  

- **Mean accuracy ≈ 0.989** → On average, the model correctly classifies about **98.9%** of the samples.  
- **Standard deviation ≈ 0.014** → The small variation means the model is **stable** and not sensitive to how the data are split.

Because accuracy is high and variance is low, the Logistic Regression model generalizes well on this dataset.  
If one fold had shown much lower accuracy, it would suggest overfitting or unbalanced data — but that is not the case here.